# [Chapter 10 — Sensitivity Analysis, §10.1] Local Sensitivity Indices for $\mathcal{R}_0$ and $I^*$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 10 — Sensitivity Analysis
**Considerations developed:** 10 (sensitivity)
**Estimated runtime:** ≤ 20s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Computes closed-form normalized sensitivity indices $S^y_p = (p/y)(\partial y/\partial p)$ for $\mathcal{R}_0 = c_I \beta \tau_R$ and $I^*$. Verifies analytical predictions against finite-difference computation.

## What you should already know
Chapter 6 R_0 notebooks; basic calculus.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_10
from shared.seeds import set_seed_chapter_10
from shared.verification import assert_within_tolerance
set_seed_chapter_10()
book_style()


## Closed-form sensitivities

For $\mathcal{R}_0 = c_I \beta \tau_R$, the normalized sensitivity index is the multiplicative role:
- $S^{R_0}_{c_I} = +1$
- $S^{R_0}_{\beta} = +1$
- $S^{R_0}_{\tau_R} = +1$
- $S^{R_0}_{c_S} = 0, \; S^{R_0}_{c_R} = 0$

For the endemic burden $I^* = (\mu N / (\mu + 1/\tau_R))(1 - 1/\mathcal{R}_0)$, the sensitivities are different — confirming Consideration 6's invasion-burden distinction.

In [ ]:
params = baseline_chapter_10()
c_I, beta, tau_R = params['c_I'], params['beta'], params['tau_R']
R0 = c_I * beta * tau_R

# Finite-difference sensitivity computation
def R0_func(c_I_, beta_, tau_R_):
    return c_I_ * beta_ * tau_R_

def fd_sensitivity(func, base_args, idx, h=1e-6):
    # Normalized sensitivity index (p/y)(dy/dp)
    args_plus = list(base_args); args_plus[idx] *= (1 + h)
    args_minus = list(base_args); args_minus[idx] *= (1 - h)
    y_plus = func(*args_plus); y_minus = func(*args_minus)
    y0 = func(*base_args)
    return (base_args[idx] / y0) * (y_plus - y_minus) / (2 * h * base_args[idx])

base = (c_I, beta, tau_R)
S_R0_cI = fd_sensitivity(R0_func, base, 0)
S_R0_beta = fd_sensitivity(R0_func, base, 1)
S_R0_tauR = fd_sensitivity(R0_func, base, 2)

print(f"Sensitivity indices for R_0 = c_I * beta * tau_R:")
print(f"  S^R0_cI    = {S_R0_cI:.6f}  (analytical: +1)")
print(f"  S^R0_beta  = {S_R0_beta:.6f}  (analytical: +1)")
print(f"  S^R0_tauR  = {S_R0_tauR:.6f}  (analytical: +1)")

assert_within_tolerance(S_R0_cI, 1.0, rel_tol=1e-4)
assert_within_tolerance(S_R0_beta, 1.0, rel_tol=1e-4)
assert_within_tolerance(S_R0_tauR, 1.0, rel_tol=1e-4)
print("\n✅ Finite-difference sensitivity matches analytical formula")
